## 04_modeling: Dự đoán supervised
Mẫu train regression/forecasting với RandomForest và LightGBM.

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load dữ liệu đã xử lý
candidates = [
    '../data/processed/weather_featured.csv',
    '../data/processed/weather_cleaned.csv',
    '../data/processed/weather_mining.csv'
]

data_path = None
for p in candidates:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError(f"Không tìm thấy file data trong {candidates}")

print(f"Đang dùng file: {data_path}")

df = pd.read_csv(data_path, parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
if 'target_variable' not in df.columns:
    raise KeyError("Cột 'target_variable' không tồn tại trong file dữ liệu")
y = df['target_variable']
X = df.drop(columns=['timestamp', 'target_variable'])

# time series split
tscv = TimeSeriesSplit(n_splits=5)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_scores = cross_val_score(rf, X, y, cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1)
print('RF RMSE CV:', -rf_scores.mean())

# LightGBM
lgb_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
lgb_scores = cross_val_score(lgb_model, X, y, cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1)
print('LGBM RMSE CV:', -lgb_scores.mean())

# Fit final model and evaluate on last split
train_idx, test_idx = list(tscv.split(X))[-1]
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
lgb_model.fit(X_train, y_train)
y_pred = lgb_model.predict(X_test)
print('Final RMSE', mean_squared_error(y_test, y_pred, squared=False))
print('Final MAE', mean_absolute_error(y_test, y_pred))